# Tutorial 3: Spec Contracts and Debugging

Estimated time: 25-35 minutes

## Prerequisites
No optional dependencies required.

## Learning aims
- Primary package aim: debug validation failures and restore a valid spec
- Secondary scientific aim: understand why explicit contracts protect scientific assumptions

## Success criteria
- you can intentionally break and repair a spec using validator output


## Why this tutorial matters
Each step connects the CLI workflow to scientific reasoning so you can explain not only *what* ran, but *why* results are meaningful.


## Step 1: Create a temporary copy and inject an error


In [ ]:
# Cross-platform setup — works on Windows / macOS / Linux.
# Locates the repo root, puts src/ on sys.path, and defines helpers that run
# the `mm` CLI in-process (run_mm_cli) and auxiliary tools like pytest/ruff
# via the active interpreter (run_tool). No shell cells, no PYTHONPATH prefix.
import io
import os
import subprocess
import sys
from contextlib import contextmanager, redirect_stderr, redirect_stdout
from pathlib import Path

root = Path.cwd().resolve()
if not (root / "src").is_dir() and (root.parent / "src").is_dir():
    root = root.parent
if not (root / "src").is_dir():
    raise RuntimeError("Could not locate project root (expected src/).")

src_path = root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# Run the whole notebook from the repo root so CLI artifacts and any
# CWD-relative registry lookups (e.g. eval_surrogate) resolve consistently.
os.chdir(root)

# Jupyter caches imported modules; clear bayesian_metamodeling so re-runs pick
# up the current local source.
for module_name in list(sys.modules):
    if module_name == "bayesian_metamodeling" or module_name.startswith("bayesian_metamodeling."):
        del sys.modules[module_name]

from bayesian_metamodeling.cli.main import main as mm_main


@contextmanager
def in_project_root():
    previous = Path.cwd()
    os.chdir(root)
    try:
        yield
    finally:
        os.chdir(previous)


def run_mm_cli(*args: str, check: bool = True) -> int:
    """Run `mm <args>` in-process; cross-platform, no shell."""
    stdout_buf = io.StringIO()
    stderr_buf = io.StringIO()
    previous_argv = sys.argv[:]
    try:
        sys.argv = ["mm", *args]
        with in_project_root(), redirect_stdout(stdout_buf), redirect_stderr(stderr_buf):
            exit_code = mm_main()
    finally:
        sys.argv = previous_argv

    print("$ mm", " ".join(args))
    out = stdout_buf.getvalue().strip()
    err = stderr_buf.getvalue().strip()
    if out:
        print(out)
    if err:
        print(err)
    if check and exit_code != 0:
        raise RuntimeError(f"CLI command failed ({exit_code}): mm {' '.join(args)}")
    return exit_code


def run_tool(*args: str, check: bool = True) -> int:
    """Run an auxiliary tool (pytest, ruff) via the active interpreter, cross-platform."""
    cmd = list(args)
    if cmd and cmd[0] in {"pytest", "ruff"}:
        cmd = [sys.executable, "-m", *cmd]
    print("$", " ".join(args))
    with in_project_root():
        result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print(result.stderr.strip())
    if check and result.returncode != 0:
        raise RuntimeError(f"Tool failed ({result.returncode}): {' '.join(args)}")
    return result.returncode


In [ ]:
import json
from pathlib import Path

root = Path.cwd().resolve()
if not (root / "src").is_dir() and (root.parent / "src").is_dir():
    root = root.parent

src = root / 'tutorials/specs/model.toy.grid.json'
dst = root / 'tmp/tutorials_modelspec_edit.json'
dst.parent.mkdir(parents=True, exist_ok=True)
dst.write_text(src.read_text())

payload = json.loads(dst.read_text())
payload['design'].pop('grid', None)
dst.write_text(json.dumps(payload, indent=2))
print('Wrote broken spec:', dst)

## Step 2: Validate and inspect actionable error


In [ ]:
run_mm_cli('validate', 'tmp/tutorials_modelspec_edit.json', check=False)


**Expected output:** The validator should report a validation error because the `grid` key was removed from the `design` section. You should see a message like:

```
Spec validation failed: 1 validation error for ModelSpec
...
  grid config required when strategy is 'grid'
```

This actionable error tells you exactly which section is broken and what contract was violated.

## Step 3: Repair and re-validate


In [ ]:
import json
from pathlib import Path

root = Path.cwd().resolve()
if not (root / "src").is_dir() and (root.parent / "src").is_dir():
    root = root.parent

p = root / 'tmp/tutorials_modelspec_edit.json'
payload = json.loads(p.read_text())
payload['design']['grid'] = {'a': [0.0, 1.0, 2.0], 'b': [0.0, 1.0, 2.0]}
p.write_text(json.dumps(payload, indent=2))
print('Spec repaired')

**Expected output:** After repair, the validator should print a success message:

```
Spec is valid.  (model='test', strategy=grid, 9 design points)
```

If you still see errors, double-check that the `grid` key maps variable names to lists of values matching the `io_schema.inputs` names.

In [ ]:
run_mm_cli('validate', 'tmp/tutorials_modelspec_edit.json')


## Scientific checkpoint
For each input variable, write:
- meaning in your domain,
- plausible support range and why,
- what scientific bias could appear if support bounds are wrong.


## Contract coverage graphic


In [ ]:
import matplotlib.pyplot as plt

sections = ['model', 'runner', 'io_schema', 'design', 'adapter', 'reproducibility', 'storage']
importance = [5, 4, 5, 5, 4, 3, 4]

plt.figure(figsize=(8, 3))
plt.bar(sections, importance, color='tab:purple')
plt.xticks(rotation=30, ha='right')
plt.ylabel('importance (relative)')
plt.title('Spec sections as scientific contract layers')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()
